# Keal Voice — Colab Voice Cloning Studio

Write a script, generate speech in your own voice, and download an MP3. This notebook runs Qwen3-TTS on the Colab GPU. Your voice profile and generated files are stored in your Google Drive; model weights are cached in the temporary Colab runtime.

**Use only your own voice or a voice you have explicit permission to use.** Do not use generated speech to impersonate someone or mislead listeners. Synthetic speech can still be detectable; naturalness is not guaranteed.

**Language note:** Qwen3-TTS officially supports English, Chinese, Japanese, Korean, German, French, Russian, Portuguese, Spanish and Italian. Hindi is not officially supported by this model.

## Run instructions
1. In Colab, choose **Runtime → Change runtime type → T4 GPU** (or another available GPU).
2. Run the cells from top to bottom.
3. Connect Google Drive when prompted.
4. In the app, upload a clean voice sample once, paste its exact transcript, and save your voice profile.
5. Enter a script and click **Generate MP3**. Your profile remains in Drive for later sessions.

A GPU is required. Free Colab GPU availability and session limits vary.

In [ ]:
# Prepare a compatible Colab environment for Qwen3-TTS
# Colab may include a newer diffusers package that requires huggingface-hub 1.x.
# Qwen3-TTS uses Transformers 4.57.3, which requires huggingface-hub < 1.0.
# diffusers is not used by this notebook, so remove it to avoid the conflict.
!pip -q uninstall -y diffusers
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip -q install "huggingface-hub>=0.34.0,<1.0" "qwen-tts==0.1.1" gradio soundfile

# Verify the key package versions before loading the model.
from importlib.metadata import version
for package in ['qwen-tts', 'transformers', 'huggingface-hub', 'accelerate', 'gradio']:
    print(f'{package}: {version(package)}')


In [ ]:
# Connect Google Drive so your voice profile and MP3 files persist between sessions.
from google.colab import drive
from pathlib import Path
import os

try:
    drive.mount('/content/drive', force_remount=True, timeout_ms=120000)
except Exception as exc:
    raise RuntimeError(
        'Google Drive could not be mounted. In Colab, use Runtime > Disconnect and delete runtime, '
        'then reopen this notebook in Chrome (not an in-app browser), allow Google sign-in/pop-ups, '
        'and run this cell again. Original error: ' + str(exc)
    ) from exc

BASE_DIR = Path('/content/drive/MyDrive/KealVoice')
PROFILE_DIR = BASE_DIR / 'profile'
OUTPUT_DIR = BASE_DIR / 'outputs'
PROFILE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROFILE_AUDIO = PROFILE_DIR / 'voice_reference'
PROFILE_TEXT = PROFILE_DIR / 'voice_reference.txt'

print('Keal Voice storage ready:', BASE_DIR)
print('Saved voice profile found:', PROFILE_TEXT.exists() and any(p.suffix.lower() != '.txt' for p in PROFILE_DIR.glob('voice_reference.*')))


In [ ]:
import torch
from qwen_tts import Qwen3TTSModel

if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. In Colab, select Runtime → Change runtime type → GPU, then rerun this cell.')

gpu_name = torch.cuda.get_device_name(0)
total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f'GPU: {gpu_name} ({total_vram:.1f} GB VRAM)')

# 1.7B is selected for quality. If it does not fit in the GPU memory available
# to your Colab session, change MODEL_ID to the 0.6B Base model below.
MODEL_ID = 'Qwen/Qwen3-TTS-12Hz-1.7B-Base'
# Lower-memory fallback: 'Qwen/Qwen3-TTS-12Hz-0.6B-Base'

model = Qwen3TTSModel.from_pretrained(
    MODEL_ID,
    device_map='cuda:0',
    dtype=torch.float16,
    attn_implementation='sdpa',
)
print('Model loaded:', MODEL_ID)

In [ ]:
import gradio as gr
import soundfile as sf
import numpy as np
import shutil
import subprocess
import tempfile
import uuid
from datetime import datetime
from pathlib import Path

LANGUAGES = [
    'English', 'Chinese', 'Japanese', 'Korean', 'German',
    'French', 'Russian', 'Portuguese', 'Spanish', 'Italian'
]

def find_saved_profile():
    files = [p for p in PROFILE_DIR.glob('voice_reference.*') if p.suffix.lower() != '.txt']
    return files[0] if files and PROFILE_TEXT.exists() else None

def save_voice_profile(audio_path, transcript, permission):
    if not permission:
        raise gr.Error('Please confirm that this is your own voice or that you have explicit permission to use it.')
    if not audio_path:
        raise gr.Error('Upload a clear recording of your voice first.')
    if not transcript or len(transcript.strip()) < 5:
        raise gr.Error('Enter the exact words spoken in the reference recording.')
    source = Path(audio_path)
    if not source.exists():
        raise gr.Error('The uploaded reference audio could not be found. Please upload it again.')
    for old in PROFILE_DIR.glob('voice_reference.*'):
        old.unlink(missing_ok=True)
    destination = PROFILE_DIR / f'voice_reference{source.suffix.lower()}'
    shutil.copy2(source, destination)
    PROFILE_TEXT.write_text(transcript.strip(), encoding='utf-8')
    return f'Voice profile saved to Google Drive. File: {destination.name}'

def make_mp3(script, language, permission):
    if not permission:
        raise gr.Error('Please confirm that you are using your own voice or have explicit permission.')
    if not script or not script.strip():
        raise gr.Error('Enter the text you want spoken.')
    if len(script) > 4000:
        raise gr.Error('For this first version, keep each script under 4,000 characters. This helps avoid long-generation memory and quality problems.')
    ref_audio = find_saved_profile()
    if ref_audio is None:
        raise gr.Error('No saved voice profile found. Upload your voice sample and transcript, then click Save voice profile.')
    ref_text = PROFILE_TEXT.read_text(encoding='utf-8').strip()
    if not ref_text:
        raise gr.Error('Your saved reference transcript is empty. Save the voice profile again.')

    try:
        wavs, sample_rate = model.generate_voice_clone(
            text=script.strip(),
            language=language,
            ref_audio=str(ref_audio),
            ref_text=ref_text,
        )
        wav = np.asarray(wavs[0], dtype=np.float32)
        stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        token = uuid.uuid4().hex[:6]
        wav_path = OUTPUT_DIR / f'keal_voice_{stamp}_{token}.wav'
        mp3_path = OUTPUT_DIR / f'keal_voice_{stamp}_{token}.mp3'
        sf.write(str(wav_path), wav, sample_rate)

        # Loudness normalization and high-quality MP3 encoding.
        command = [
            'ffmpeg', '-y', '-i', str(wav_path),
            '-af', 'loudnorm=I=-16:TP=-1.5:LRA=11',
            '-ar', '44100', '-codec:a', 'libmp3lame', '-b:a', '192k',
            str(mp3_path)
        ]
        result = subprocess.run(command, capture_output=True, text=True)
        if result.returncode != 0:
            raise RuntimeError(result.stderr[-1800:])
        return str(mp3_path), str(mp3_path), f'Generated and saved: {mp3_path.name}'
    except Exception as exc:
        raise gr.Error(f'Generation failed: {exc}')

saved = find_saved_profile()
profile_status = 'A saved voice profile is ready.' if saved else 'No voice profile saved yet. Upload a reference recording below.'

with gr.Blocks(title='Keal Voice — AI Voice Studio', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# Keal Voice\nWrite a script and generate an MP3 in your own voice.')
    gr.Markdown('Your voice profile and generated MP3 files are stored in your Google Drive folder: `MyDrive/KealVoice`. Do not upload someone else’s voice without permission.')
    status = gr.Markdown(profile_status)
    with gr.Accordion('1. Set up or replace your voice profile', open=not bool(saved)):
        ref_audio_input = gr.Audio(label='Upload a clean recording of your voice', type='filepath')
        ref_transcript = gr.Textbox(label='Exact transcript of the recording', lines=3, placeholder='Type exactly what you say in the recording...')
        permission_save = gr.Checkbox(label='I confirm this is my own voice or I have explicit permission to use it.', value=False)
        save_btn = gr.Button('Save voice profile', variant='secondary')
        save_status = gr.Markdown()
        save_btn.click(save_voice_profile, inputs=[ref_audio_input, ref_transcript, permission_save], outputs=save_status)
    gr.Markdown('### 2. Generate speech')
    script_input = gr.Textbox(label='Your script', lines=8, placeholder='Write anything you want your voice to say...', max_lines=18)
    language_input = gr.Dropdown(choices=LANGUAGES, value='English', label='Speech language')
    permission_generate = gr.Checkbox(label='I confirm that this generation uses my own voice or a voice I am authorized to use.', value=False)
    generate_btn = gr.Button('Generate MP3', variant='primary')
    audio_preview = gr.Audio(label='Generated MP3 preview', type='filepath')
    download_file = gr.File(label='Download MP3')
    generation_status = gr.Markdown()
    generate_btn.click(make_mp3, inputs=[script_input, language_input, permission_generate], outputs=[audio_preview, download_file, generation_status])
    gr.Markdown('**Quality note:** Clear reference audio and an accurate transcript improve cloning. MP3 encoding cannot repair poor synthesis; always listen to the result before using it.')

# Colab notebooks need a public Gradio tunnel; localhost is not reachable from your phone browser.
import secrets
GRADIO_USER = 'keal'
GRADIO_PASSWORD = secrets.token_urlsafe(12)
print('Keal Voice login username:', GRADIO_USER)
print('Keal Voice login password:', GRADIO_PASSWORD)
print('Keep these credentials private. Anyone with the link and credentials can access this running app.')
demo.launch(share=True, debug=True, auth=(GRADIO_USER, GRADIO_PASSWORD))